# NumCompute Quickstart: Exploring Titanic Passengers

Instead of working with a table of unexplained prices, this quickstart uses passenger records with familiar fields such as age, fare, sex, and survival. The CSV contains real missing values, so we can see why preprocessing is useful.

We will load and inspect the records, summarize and visualize passenger ages, then use a pipeline to fill in missing ages/fares and put them on a comparable scale. We will also encode a category and evaluate a simple, clearly stated rule. At the end, short examples will introduce the rest of NumCompute: streaming, ranking, sorting, numeric utilities, derivatives, and benchmarking.

Dataset: [Seaborn Titanic dataset](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv), included locally as `demo/data/titanic.csv` so the notebook does not need to download data while running.

The setup cell finds the project folder and imports NumCompute directly from its source; it does not depend on a `.venv` folder. The selected Python kernel still needs NumPy installed.

In [1]:
from pathlib import Path
import sys

import numpy as np

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "NumCompute" / "numcompute").is_dir()),
    Path.cwd(),
)
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "NumCompute"))

from numcompute.io import iter_csv, load_csv
from numcompute.histogram_display import display_histogram
from numcompute.metrics import accuracy, auc, confusion_matrix, f1, mse, precision, recall, roc_curve
from numcompute.optim import grad, jacobian
from numcompute.pipeline import Compose, FeatureUnion, Pipeline
from numcompute.preprocessing import Imputer, MinMaxScaler, OneHotEncoder, StandardScaler
from numcompute.rank import Ranker
from numcompute.sort_search import binary_search, multi_key_sort, quickselect, stable_sort, topk
from numcompute.stats import histogram, maximum, mean, median, minimum, percentile, quantile, std
from numcompute.utils import batches, cosine_similarity, euclidean_distance, logsumexp, manhattan_distance, relu, sigmoid, softmax, topk_indices
from numcompute.benchmarking import benchmark, compare, format_table

DATA_PATH = project_root / "demo" / "data" / "titanic.csv"
print("Dataset:", DATA_PATH)

Dataset: /Users/prajalpatidar/Desktop/COMP-5004/Morpheus-Assignment-2.1-Programming-Task-1-/demo/data/titanic.csv


## 1. Load the passenger records

`load_csv` returns the column names and the table. We ask it to represent blank cells as `NaN` (not a made-up value). `NaN` means the original record did not include that information.

In [2]:
columns, passengers = load_csv(DATA_PATH, fill_value=np.nan)
column_index = {name: index for index, name in enumerate(columns)}

print(f"Loaded {passengers.shape[0]} passenger records.")
print("Available columns:", ", ".join(columns))

shown_columns = ["survived", "pclass", "sex", "age", "fare", "embarked"]
print("\nFirst five passengers (survived: 1 = yes, 0 = no):")
for row in passengers[:5]:
    print({name: row[column_index[name]] for name in shown_columns})

Loaded 891 passenger records.
Available columns: survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone

First five passengers (survived: 1 = yes, 0 = no):
{'survived': 0.0, 'pclass': 3.0, 'sex': 'male', 'age': 22.0, 'fare': 7.25, 'embarked': 'S'}
{'survived': 1.0, 'pclass': 1.0, 'sex': 'female', 'age': 38.0, 'fare': 71.2833, 'embarked': 'C'}
{'survived': 1.0, 'pclass': 3.0, 'sex': 'female', 'age': 26.0, 'fare': 7.925, 'embarked': 'S'}
{'survived': 1.0, 'pclass': 1.0, 'sex': 'female', 'age': 35.0, 'fare': 53.1, 'embarked': 'S'}
{'survived': 0.0, 'pclass': 3.0, 'sex': 'male', 'age': 35.0, 'fare': 8.05, 'embarked': 'S'}


## 2. Understand the records

`age` and `fare` are measurements. `survived` is an outcome: `1` means the passenger survived and `0` means they did not. We report ages in years and fares in the dataset's currency units, rather than printing an unlabeled block of statistics.

In [3]:
ages = passengers[:, column_index["age"]].astype(float)
fares = passengers[:, column_index["fare"]].astype(float)
survived = passengers[:, column_index["survived"]].astype(int)

known_ages = ages[~np.isnan(ages)]
known_fares = fares[~np.isnan(fares)]
missing_ages = int(np.isnan(ages).sum())
missing_fares = int(np.isnan(fares).sum())

print(f"Passengers with a recorded age: {len(known_ages)} of {len(ages)}")
print(f"Missing ages: {missing_ages}; missing fares: {missing_fares}")
print(f"Average recorded age: {mean(known_ages):.1f} years")
print(f"Typical recorded age (median): {median(known_ages):.1f} years")
print(f"Age spread (standard deviation): {std(known_ages):.1f} years")
print(f"Passengers who survived: {survived.sum()} of {len(survived)} ({100 * mean(survived):.1f}%)")

Passengers with a recorded age: 714 of 891
Missing ages: 177; missing fares: 0
Average recorded age: 29.7 years
Typical recorded age (median): 28.0 years
Age spread (standard deviation): 14.5 years
Passengers who survived: 342 of 891 (38.4%)


## 3. Visualize passenger ages

The histogram groups known ages into ranges. Longer bars mean more passengers were in that age range. Missing ages are excluded from this chart; we will handle them in the next section.

In [4]:
age_counts, age_edges = histogram(known_ages, bins=10)
display_histogram(age_counts, age_edges)

True

## 4. Fill missing values and prepare features with a pipeline

A pipeline runs the listed steps in order on the same data:

1. **Imputer:** replace a missing age or fare with the median of that column. In this CSV, 177 ages are missing and no fares are missing, so the age values are the ones that actually get filled. The median is a typical passenger value and is less affected by unusually high fares than the mean.
2. **StandardScaler:** convert each column to a relative score. It subtracts that column's average and divides by its spread. After this step, age is no longer measured in years and fare is no longer in currency units. A negative score is below that column's average; a positive score is above it.

We show an individual missing age before and after filling, then the final scaled age and fare. The original CSV is not changed.

In [5]:
numeric_features = np.column_stack((ages, fares))
feature_pipeline = Pipeline([
    ("fill_missing_values", Imputer(strategy="median")),
    ("standardize_features", StandardScaler()),
])
prepared_features = feature_pipeline.fit_transform(numeric_features)

# Fit an individual age imputer too, so the filled values are easy to read.
age_imputer = Imputer(strategy="median")
filled_ages = age_imputer.fit_transform(ages)

print(f"Median age used for missing ages: {age_imputer.statistics_:.1f} years")
print(f"Missing fares: {missing_fares} (so no fare values needed filling)")
print(f"Missing numeric values after the pipeline: {int(np.isnan(prepared_features).sum())}")
print("\nPassenger row | age before | age after filling | scaled age | fare before | scaled fare")
for row_index in [0, 5, 1]:
    age_before = "missing" if np.isnan(ages[row_index]) else f"{ages[row_index]:.1f}"
    fare_before = "missing" if np.isnan(fares[row_index]) else f"{fares[row_index]:.2f}"
    print(
        f"{row_index + 1:12} | {age_before:10} | {filled_ages[row_index]:16.1f} | "
        f"{prepared_features[row_index, 0]:10.2f} | {fare_before:11} | "
        f"{prepared_features[row_index, 1]:10.2f}"
    )

Median age used for missing ages: 28.0 years
Missing fares: 0 (so no fare values needed filling)
Missing numeric values after the pipeline: 0

Passenger row | age before | age after filling | scaled age | fare before | scaled fare
           1 | 22.0       |             22.0 |      -0.57 | 7.25        |      -0.50
           6 | missing    |             28.0 |      -0.10 | 8.46        |      -0.48
           2 | 38.0       |             38.0 |       0.66 | 71.28       |       0.79


## 5. Turn a passenger category into usable features

Computers need a numeric representation of labels such as `female` and `male`. `OneHotEncoder` gives each label its own 0/1 column. These values are indicators, not measurements: `[1, 0]` means female and `[0, 1]` means male in the category order shown below.

In [6]:
sex = passengers[:, column_index["sex"]].astype(str)
sex_categories = np.unique(sex)
encoded_sex = OneHotEncoder().fit_transform(sex)

print("Encoded column order:", sex_categories.tolist())
for row_index in range(5):
    print(f"{sex[row_index]:6} -> {encoded_sex[row_index].astype(int).tolist()}")

Encoded column order: ['female', 'male']
male   -> [0, 1]
female -> [1, 0]
female -> [1, 0]
female -> [1, 0]
male   -> [0, 1]


## 6. Read evaluation results in context

To demonstrate classification metrics without pretending we trained a model, use a deliberately simple rule: **predict that a passenger survived if the record says female**. The metrics compare those rule-based guesses with the actual `survived` column. They evaluate this rule; they do not explain why any individual passenger survived.

In [7]:
predicted_survival = (sex == "female").astype(int)
matrix = confusion_matrix(survived, predicted_survival)

print("Rule: predict survival for female passengers")
print(f"Correct predictions: {accuracy(survived, predicted_survival) * 100:.1f}%")
print(f"Of passengers predicted to survive, actually survived (precision): {precision(survived, predicted_survival) * 100:.1f}%")
print(f"Of passengers who survived, found by the rule (recall): {recall(survived, predicted_survival) * 100:.1f}%")
print(f"Combined precision/recall score (F1): {f1(survived, predicted_survival):.2f}")
print("\nConfusion matrix (rows = actual [did not survive, survived]; columns = predicted [did not survive, survived]):")
print(matrix)

Rule: predict survival for female passengers
Correct predictions: 78.7%
Of passengers predicted to survive, actually survived (precision): 74.2%
Of passengers who survived, found by the rule (recall): 68.1%
Combined precision/recall score (F1): 0.71

Confusion matrix (rows = actual [did not survive, survived]; columns = predicted [did not survive, survived]):
[[468  81]
 [109 233]]


## What changed?

- Blank ages and fares were visible as missing values instead of being silently replaced with zero.
- The pipeline replaced each missing numeric value with that feature's median, then standardized age and fare separately.
- Encoded category values now have clear labels, and the evaluation numbers describe a specific passenger rule.

This keeps the workflow grounded in the passenger records: load → inspect → summarize → visualize → prepare → interpret.

## 7. Read larger CSV files in chunks

`load_csv` is convenient when the whole table fits in memory. `iter_csv` is for processing a file a piece at a time. Each item contains the same header and one chunk of rows; here we count the passengers without combining the chunks into another full table.

In [8]:
chunk_sizes = []
streamed_passengers = 0
for chunk_columns, chunk in iter_csv(DATA_PATH, chunksize=300, fill_value=np.nan):
    streamed_passengers += chunk.shape[0]
    chunk_sizes.append(chunk.shape[0])

print("Column names match the loaded table:", chunk_columns == columns)
print("Rows read:", streamed_passengers)
print("Rows per chunk:", chunk_sizes)

Column names match the loaded table: True
Rows read: 891
Rows per chunk: [300, 300, 291]


## 8. Ask more questions of the data

The descriptive statistics include minimum, maximum, quantile, and percentile functions. Quantiles use fractions from 0 to 1, while percentiles use values from 0 to 100. These both describe the recorded ages, so missing ages are left out before calculation.

In [9]:
print(f"Recorded age range: {minimum(known_ages):.1f} to {maximum(known_ages):.1f} years")
print(f"Middle half of ages (25th–75th percentiles): {percentile(known_ages, 25):.1f} to {percentile(known_ages, 75):.1f} years")
print(f"90th percentile age: {quantile(known_ages, 0.9):.1f} years")
print(f"Sample standard deviation: {std(known_ages, ddof=1):.1f} years")

Recorded age range: 0.4 to 80.0 years
Middle half of ages (25th–75th percentiles): 20.1 to 38.0 years
90th percentile age: 50.0 years
Sample standard deviation: 14.5 years


## 9. Rank, sort, and search passenger values

These functions answer related but different questions: sorting returns values in order, ranking assigns a position to every value, and quickselect finds one requested position without sorting the whole list. `topk` selects a few extremes (without promising to sort that selection), while binary search finds an insertion point in an already sorted array. The multi-key example orders a small passenger sample by ticket class, then fare.

In [10]:
example_ages = np.array([22.0, 38.0, 22.0, 35.0, 28.0, 35.0])
ranker = Ranker()
sorted_ages = stable_sort(example_ages)

print("Sorted ages:", sorted_ages)
print("Average ranks (ties share a rank):", ranker.rank(example_ages, method="average"))
print("Median from ranker:", ranker.percentile(example_ages, 50))
print("Third-smallest age:", quickselect(example_ages, k=3))
print("Two largest ages:", example_ages[topk(example_ages, k=2, return_indices=True)])
print("Where would age 30 be inserted?", binary_search(sorted_ages, 30.0))

sample_rows = passengers[:8]
order = multi_key_sort((sample_rows[:, column_index["pclass"]].astype(float),
                        sample_rows[:, column_index["fare"]].astype(float)))
print("First eight passenger rows ordered by class, then fare:", order)

Sorted ages: [22. 22. 28. 35. 35. 38.]
Average ranks (ties share a rank): [1.5 6.  1.5 4.5 3.  4.5]
Median from ranker: 31.5
Third-smallest age: 28.0
Two largest ages: [35. 38.]
Where would age 30 be inserted? (3, False)
First eight passenger rows ordered by class, then fare: [6 3 1 0 2 4 5 7]


## 10. Compare scaling choices and combine feature branches

`StandardScaler` expresses values relative to the average and spread. `MinMaxScaler` maps a feature into a chosen range, here 0 to 1. `FeatureUnion` runs independent transformers on the same input and joins their outputs side by side. This small example uses two simple column selectors to keep the output easy to read.

In [11]:
fare_sample = known_fares[:5]
standard_fares = StandardScaler().fit_transform(fare_sample)
scaled_fares = MinMaxScaler().fit_transform(fare_sample)
print("Fare values:", fare_sample)
print("Standardized fares:", np.round(standard_fares, 2))
print("Fares scaled to 0–1:", np.round(scaled_fares, 2))

class SelectColumn:
    def __init__(self, index):
        self.index = index

    def fit(self, X):
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.index:self.index + 1]

complete_rows = numeric_features[~np.isnan(numeric_features).any(axis=1)][:5]
feature_union = FeatureUnion([
    ("age_column", SelectColumn(0)),
    ("fare_column", SelectColumn(1)),
])
joined_features = feature_union.fit_transform(complete_rows)
print("FeatureUnion output (age and fare side by side):")
print(joined_features)

class AddOffset:
    def __init__(self, amount):
        self.amount = amount

    def fit(self, X):
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float) + self.amount

class DoubleValues:
    def fit(self, X):
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float) * 2

sequential_steps = Compose([("add one", AddOffset(1)), ("double", DoubleValues())])
print("Compose applies named steps in order:")
print(sequential_steps.fit_transform([[1, 2], [3, 4]]))

Fare values: [ 7.25   71.2833  7.925  53.1     8.05  ]
Standardized fares: [-0.82  1.53 -0.79  0.86 -0.79]
Fares scaled to 0–1: [0.   1.   0.01 0.72 0.01]
FeatureUnion output (age and fare side by side):
[[22.      7.25  ]
 [38.     71.2833]
 [26.      7.925 ]
 [35.     53.1   ]
 [35.      8.05  ]]
Compose applies named steps in order:
[[ 4.  6.]
 [ 8. 10.]]


## 11. Use NumCompute's numeric utilities

The utility module includes vector distances and similarity, common activation functions, stable probability calculations, top-k index selection, and batching. The values below are just a tiny example: softmax turns each row of scores into probabilities that add to one, and batching splits the passenger table into manageable groups.

In [12]:
first = np.array([1.0, 2.0, 3.0])
second = np.array([2.0, 2.0, 5.0])
print("Euclidean distance:", euclidean_distance(first, second))
print("Manhattan distance:", manhattan_distance(first, second))
print("Cosine similarity:", round(cosine_similarity(first, second), 3))

scores = np.array([[-1.0, 0.0, 2.0], [2.0, 1.0, -1.0]])
print("ReLU:", relu(scores[0]))
print("Sigmoid:", np.round(sigmoid(scores[0]), 3))
print("Softmax probabilities by row:\n", np.round(softmax(scores, axis=1), 3))
print("Log of summed exponentials:", round(float(logsumexp(scores[0])), 3))
print("Indices of the two largest scores:", topk_indices(scores[0], k=2))
print("Passenger chunk sizes:", [batch.shape[0] for batch in batches(passengers, 300)])

Euclidean distance: 2.23606797749979
Manhattan distance: 3.0
Cosine similarity: 0.977
ReLU: [0. 0. 2.]
Sigmoid: [0.269 0.5   0.881]
Softmax probabilities by row:
 [[0.042 0.114 0.844]
 [0.705 0.259 0.035]]
Log of summed exponentials: 2.17
Indices of the two largest scores: [2 1]
Passenger chunk sizes: [300, 300, 291]


## 12. Evaluate scores and estimate derivatives

The ROC curve shows how the female-survival rule trades off true-positive and false-positive rates as its score threshold changes. Its area (AUC) gives one summary of that ranking ability. The derivative examples are separate from the passenger analysis: they show how NumCompute estimates the slope of a function and the Jacobian of a vector-valued function using small input changes.

In [13]:
false_positive_rate, true_positive_rate, thresholds = roc_curve(
    survived, predicted_survival
)
print("ROC thresholds:", thresholds)
print("ROC AUC for the simple rule:", round(auc(false_positive_rate, true_positive_rate), 3))
print("Example mean squared error:", round(mse([1.0, 3.0, 2.0], [1.0, 2.0, 2.0]), 3))

slope = grad(lambda value: value**2, 3.0)
jacobian_matrix = jacobian(
    lambda point: np.array([point[0] ** 2 + point[1], point[0] * point[1]]),
    [2.0, 3.0],
)
print("Estimated slope of x² at x=3:", round(slope, 3))
print("Estimated Jacobian at [2, 3]:\n", np.round(jacobian_matrix, 3))

ROC thresholds: [inf  1.  0.]
ROC AUC for the simple rule: 0.767
Example mean squared error: 0.333
Estimated slope of x² at x=3: 6.0
Estimated Jacobian at [2, 3]:
 [[4. 1.]
 [3. 2.]]


## 13. Measure performance

The comparison below uses the same seeded input for a Python loop and a NumPy-vectorised calculation, verifies their outputs match, then reports median and minimum timings plus environment details. Use `python NumCompute/numcompute/benchmarking.py` to run the same benchmark outside this notebook; measurements are local and vary by machine and system load.

In [14]:
from numcompute.benchmarking import numpy_vectorized, python_loop

benchmark_data = np.random.default_rng(42).random(200_000)
np.testing.assert_allclose(python_loop(benchmark_data), numpy_vectorized(benchmark_data))

comparison = compare(
    {"Python loop": python_loop, "NumPy vectorised": numpy_vectorized},
    benchmark_data,
    repeats=5,
    warmup=1,
)
print(format_table(comparison))
print(
    "Median speedup: "
    f"{comparison['Python loop']['median_seconds'] / comparison['NumPy vectorised']['median_seconds']:.2f}x"
)

Implementation   | Median (s) | Minimum (s)
-------------------------------------------
Python loop      | 0.0163793 | 0.016203
NumPy vectorised | 9.7125e-05 | 9.5834e-05
Environment: python=3.13.9, numpy=2.3.5, platform=macOS-27.0.1-arm64-arm-64bit-Mach-O, processor=arm, cpu_count=10
Median speedup: 168.64x


## Where to go next

This notebook introduced each major area of NumCompute with small, runnable examples. The library is built around NumPy arrays: inspect the input and output shapes, read each function's docstring for options, and adapt the examples to your own data. For a larger CSV, iterate with `iter_csv`; for repeatable data preparation, keep the fitted steps together in a `Pipeline`; and keep calculations separate from presentation.